In [0]:
%sql
DROP CATALOG IF EXISTS `01_bronze` CASCADE;
DROP CATALOG IF EXISTS `01_silver` CASCADE;
DROP CATALOG IF EXISTS `01_gold` CASCADE;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS `01_bronze`
    MANAGED LOCATION 's3://electronicsretailersstorage/bronze';
CREATE CATALOG IF NOT EXISTS `02_silver`
    MANAGED LOCATION 's3://electronicsretailersstorage/silver'; 
CREATE CATALOG IF NOT EXISTS `03_gold`
    MANAGED LOCATION 's3://electronicsretailersstorage/gold';
CREATE SCHEMA IF NOT EXISTS `01_bronze`.`raw`;
CREATE SCHEMA IF NOT EXISTS `02_silver`.`transformation`;
CREATE SCHEMA IF NOT EXISTS `03_gold`.`dim_tables`;
CREATE SCHEMA IF NOT EXISTS `03_gold`.`fact_tables`;

In [0]:

def standardize_col_name(col_name):
    return col_name.strip().replace(' ', '_').lower()

def rename_columns(df):
    old_columns = df.columns
    new_columns = [standardize_col_name(col) for col in old_columns]
    renamed_df = df.toDF(*new_columns)
    return renamed_df

data = spark.read.format("csv").option("header", "true").load("s3://electronicsretailersraw/rawdata_electronics_retailers/metadata.csv")
for d in data.collect():
    temp = spark.read.format('csv').option("header","true").load(f"s3://electronicsretailersraw/rawdata_electronics_retailers/{d.file_name}.{d.extension}")
    temp = rename_columns(temp)
    temp.write.mode("overwrite").format("delta").saveAsTable(f"01_bronze.{d.schema_name}.{d.table_name}")